# Label stats

Counts boxes in `labels/` using class names from `classes.txt`.

In [1]:
from pathlib import Path
from collections import Counter

# notebook is in task3/ — fall back if cwd is the repo root
ROOT = Path.cwd()
if not (ROOT / "classes.txt").exists():
    ROOT = ROOT / "task3"

LABELS = ROOT / "labels"
CLASSES = ROOT / "classes.txt"
IMAGES = ROOT / "images"

assert CLASSES.exists(), f"can't find classes.txt under {ROOT}"
assert LABELS.exists(), f"can't find labels/ under {ROOT}"

names = [line.strip() for line in CLASSES.read_text().splitlines() if line.strip()]
names


['direction', 'give way', 'other', 'pedestrian', 'speed', 'stop']

In [2]:
counts = Counter()
boxes_per_image = []
empty = 0
missing_images = 0

label_files = sorted(LABELS.glob("*.txt"))
for label_file in label_files:
    lines = [ln for ln in label_file.read_text().splitlines() if ln.strip()]
    boxes_per_image.append(len(lines))
    if not lines:
        empty += 1

    if not any((IMAGES / f"{label_file.stem}{ext}").exists() for ext in (".jpg", ".png", ".jpeg", ".JPG")):
        missing_images += 1

    for line in lines:
        class_id = int(line.split()[0])
        counts[class_id] += 1

total_boxes = sum(counts.values())
multi = sum(1 for n in boxes_per_image if n > 1)

print(f"label files: {len(label_files)}")
print(f"empty labels: {empty}")
print(f"images with >1 box: {multi}")
print(f"labels with no matching image: {missing_images}")
print(f"total boxes: {total_boxes:,}")
print()
print("class distribution:")
for class_id, name in enumerate(names):
    n = counts[class_id]
    pct = (100 * n / total_boxes) if total_boxes else 0
    print(f"  {class_id} {name:12s}  {n:5,}  ({pct:5.1f}%)")

unknown = {k: v for k, v in counts.items() if k < 0 or k >= len(names)}
if unknown:
    print("\nunknown class ids:", dict(unknown))

label files: 628
empty labels: 32
images with >1 box: 331
labels with no matching image: 0
total boxes: 1,359

class distribution:
  0 direction       249  ( 18.3%)
  1 give way         55  (  4.0%)
  2 other           412  ( 30.3%)
  3 pedestrian      125  (  9.2%)
  4 speed           510  ( 37.5%)
  5 stop              8  (  0.6%)
